# Classification - Credit Card Fraud

**Goal:** predict `Class` (1 = fraud, 0 = legit) using KNN, Logistic Regression, Naive Bayes, Decision Tree and SVM.

The data is imbalanced, so we balance the **training set only**. The test set stays untouched, so the results are realistic. `Time` and `Amount` are already scaled in preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

data = pd.read_csv('creditcard_clean.csv')
print(data['Class'].value_counts())

## 1. Train-test split
`stratify=y` keeps the same fraud ratio in train and test.

In [ ]:
x = data.drop(columns=['Class', 'Hour'], errors='ignore')
y = data['Class']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(x_train.shape, x_test.shape)

## 2. Balance the training data
Keep all fraud rows and take the same number of random legit rows (undersampling).

In [ ]:
train = x_train.copy()
train['Class'] = y_train

class_1 = train[train['Class'] == 1]
class_0 = train[train['Class'] == 0].sample(len(class_1), random_state=42)

train_bal = pd.concat([class_0, class_1])
x_bal = train_bal.drop('Class', axis=1)
y_bal = train_bal['Class']

print(y_bal.value_counts())

## 3. Train all models and compare
- **Precision:** of all predicted frauds, how many were real
- **Recall:** of all real frauds, how many we caught (most important here)
- **F1:** balance of precision and recall

In [ ]:
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(),
}

rows = []
for name, model in models.items():
    model.fit(x_bal, y_bal)
    y_pred = model.predict(x_test)

    print(name)
    print(confusion_matrix(y_test, y_pred))

    rows.append([name, accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),
                 recall_score(y_test, y_pred), f1_score(y_test, y_pred)])

results = pd.DataFrame(rows, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1']).set_index('Model')
results.round(4)

In each confusion matrix, rows are the actual class and columns are the predicted class:
`[[TN, FP], [FN, TP]]`. **FN** = missed frauds, **FP** = legit flagged as fraud.

In [ ]:
results[['Precision', 'Recall', 'F1']].plot(kind='bar', figsize=(10, 4))
plt.title('Model comparison')
plt.xticks(rotation=0)
plt.show()

## 4. Decision tree (top 3 levels)

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(models['Decision Tree'], feature_names=list(x_bal.columns),
          class_names=['Legit', 'Fraud'], filled=True, max_depth=3)
plt.show()

## Conclusion
- Accuracy is not used to pick the best model, because the test data is imbalanced.
- Pick the model with the best **recall** and a reasonable **precision** (check the F1 column).
- Because we balanced the training data, precision is low for every model: many legit transactions get flagged as fraud.